In [1]:
# ===== 라이브러리 =====
import numpy as np
import torch
from datasets import load_dataset                       # KLUE 데이터를 한 줄로 가져오는 도구
from transformers import (
    AutoTokenizer,                                       # 글자 → 숫자 변환기
    AutoModelForSequenceClassification,                  # BERT + 분류 머리
    TrainingArguments, Trainer,                          # 학습 자동화
    DataCollatorWithPadding,                             # 배치마다 동적 패딩
    set_seed,
)
from sklearn.metrics import accuracy_score, f1_score     # 채점용

# ★ 쇼핑 때랑 비교: pandas, urllib 이 사라졌어요!
#   YNAT는 raw txt를 직접 안 받고 load_dataset이 알아서 가져오니까
#   표 다루는 pandas, 다운로드용 urllib 이 필요 없어요. 그만큼 데이터 단계가 간단해짐.

SEED = 42
set_seed(SEED)                                           # 매번 같은 결과 (재현성)

# 4090(Ada)에서 행렬연산 가속 — 정확도 거의 그대로, 속도 ↑
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
# KLUE 벤치마크의 ynat(뉴스 토픽 분류) 데이터를 통째로 가져옴
raw = load_dataset("klue", "ynat")

# ★ 이 한 줄이 쇼핑 때의 [다운로드 → pandas 읽기 → 라벨 만들기 → split] 4단계를 다 대신함!
#   - 이미 train / validation 으로 나뉘어서 옴 (split 안 해도 됨)
#   - 라벨도 0~6 정수로 박혀서 옴 (평점→라벨 변환 안 해도 됨)
print(raw)

# 데이터 한 개 직접 들여다보기 — 어떻게 생겼나 눈으로 확인
print(raw["train"][0])

# ★ 라벨 이름(클래스 7개)을 데이터에서 직접 꺼내옴
#   순서를 내가 외워서 쓰는 게 아니라, 데이터가 알려주는 정답을 그대로 받는 게 안전해요
label_names = raw["train"].features["label"].names
NUM_LABELS = len(label_names)
print(f"\n클래스 {NUM_LABELS}개:", label_names)
#   예: ['IT과학', '경제', '사회', '생활문화', '세계', '스포츠', '정치']

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})
{'guid': 'ynat-v1_train_00000', 'title': '유튜브 내달 2일까지 크리에이터 지원 공간 운영', 'label': 3, 'url': 'https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=105&sid2=227&oid=001&aid=0008508947', 'date': '2016.06.30. 오전 10:36'}

클래스 7개: ['IT과학', '경제', '사회', '생활문화', '세계', '스포츠', '정치']


In [3]:
MODEL_NAME = "klue/bert-base"
MAX_LEN = 64                          # ★ 쇼핑(128)보다 짧게! 뉴스 '제목'이라 짧거든요. 메모리·속도 ↓

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    # ★ 쇼핑은 "review" 컬럼이었는데, YNAT는 "title"(뉴스 제목) 컬럼!
    return tokenizer(batch["title"], truncation=True, max_length=MAX_LEN)

# train / validation 둘 다 토큰화
tokenized = raw.map(tokenize, batched=True)

# 배치 안에서 제일 긴 문장에만 맞춰 패딩 → 효율적
collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [4]:
# num_labels 를 ②에서 구한 NUM_LABELS(=7)로 자동 지정
#  ★ 쇼핑은 num_labels=2(긍/부정)였는데, 여기선 7개 토픽 중 하나 고르기
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
)
# (여기서 "classifier 새로 초기화됐다"는 그 리포트 또 뜰 거예요 — 정상! 분류 머리 새로 다는 중)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        # ★ 핵심: average="macro"
        #   클래스가 7개라서, 각 클래스의 f1을 따로 구한 뒤 단순 평균을 냄.
        #   → 데이터 적은 클래스도 똑같이 한 표! (많은 클래스에 묻히지 않게)
        #   YNAT 공식 평가지표가 바로 이 macro F1 이에요.
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [6]:
args = TrainingArguments(
    output_dir="./ynat_out",
    per_device_train_batch_size=64,    # 4090 24GB면 충분. 제목이 짧아 더 여유로움
    per_device_eval_batch_size=128,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,                  # ★ warmup_ratio 대신 warmup_steps (오늘 본 deprecated 경고 회피)
    bf16=True,                         # 4090도 bf16 OK
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",  # ★ best 기준도 macro_f1 으로!
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
)

In [7]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],  # ★ YNAT는 test 라벨이 숨겨져 있어서(벤치마크라!) validation으로 평가
    processing_class=tokenizer,            # ★ 오늘 배운 거! tokenizer= 가 아니라 processing_class=
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()   # 🚀 학습 시작! 4090이면 몇 분이면 끝나요

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.350000,0.386020,0.861864,0.864166
2,0.279100,0.358722,0.869441,0.871800
3,0.218800,0.366140,0.871088,0.872458


TrainOutput(global_step=2142, training_loss=0.3592989023993997, metrics={'train_runtime': 94.4009, 'train_samples_per_second': 1451.617, 'train_steps_per_second': 22.69, 'total_flos': 1592514152074800.0, 'train_loss': 0.3592989023993997, 'epoch': 3.0})

In [8]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],  # ★ YNAT는 test 라벨이 숨겨져 있어서(벤치마크라!) validation으로 평가
    processing_class=tokenizer,            # ★ 오늘 배운 거! tokenizer= 가 아니라 processing_class=
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()   # 🚀 학습 시작! 4090이면 몇 분이면 끝나요

Epoch,Training Loss,Validation Loss


TrainOutput(global_step=2142, training_loss=0.12985156163448044, metrics={'train_runtime': 90.2969, 'train_samples_per_second': 1517.594, 'train_steps_per_second': 23.722, 'total_flos': 1592514152074800.0, 'train_loss': 0.12985156163448044, 'epoch': 3.0})

In [9]:
save_path = "./ynat_out/best"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("저장 완료 →", save_path)

저장 완료 → ./ynat_out/best


In [10]:
def predict(titles):
    model.eval()
    enc = tokenizer(titles, padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)
    preds = probs.argmax(dim=-1)
    for t, p, pr in zip(titles, preds, probs):
        # ★ label_names 로 숫자 → 토픽 이름 변환
        print(f"[{label_names[p.item()]}] (확률 {pr[p].item():.3f})  {t}")

predict([
    "코스피 사상 최고치 경신…외국인 순매수 행진",
    "손흥민 멀티골…토트넘 3대1 완승",
    "정부 내년 예산안 발표, 복지 예산 대폭 확대",
    "신형 아이폰 공개…AI 기능 대거 탑재",
])

[경제] (확률 0.999)  코스피 사상 최고치 경신…외국인 순매수 행진
[스포츠] (확률 0.999)  손흥민 멀티골…토트넘 3대1 완승
[정치] (확률 0.995)  정부 내년 예산안 발표, 복지 예산 대폭 확대
[IT과학] (확률 0.995)  신형 아이폰 공개…AI 기능 대거 탑재


In [13]:
import os
print("현재 폴더:", os.getcwd())

# 저장 폴더 찾기 — 이름에 ynat 들어간 거 다 찾기
!find / -name "*.safetensors" 2>/dev/null
!ls -la

현재 폴더: /root


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/root/.cache/huggingface/hub/models--klue--bert-base/snapshots/77c8b3d707df785034b4e50f2da5d37be5f0f546/model.safetensors
/root/ynat_out/checkpoint-714/model.safetensors
/root/ynat_out/checkpoint-1428/model.safetensors
/root/ynat_out/checkpoint-2142/model.safetensors
/root/ynat_out/best/model.safetensors
total 16
drwx------ 1 root root 4096 May 21 05:09 .
drwxr-xr-x 1 root root  124 May 21 04:57 ..
-rw-r--r-- 1 root root 3309 May 21 04:57 .bashrc
drwxr-xr-x 1 root root  101 May 21 05:07 .cache
drwx------ 3 root root   17 May 21 05:00 .copilot
drwxr-xr-x 3 root root   28 May 21 05:00 .dotnet
drwxr-xr-x 3 root root   29 May 21 05:02 .ipython
drwxr-xr-x 3 root root   95 Sep 24  2024 .jupyter
drwx------ 3 root root   39 Sep 24  2024 .launchpadlib
drwxr-xr-x 1 root root   19 Sep 24  2024 .local
drwx------ 3 root root   26 May 21 05:09 .nv
-rw-r--r-- 1 root root  161 Jul  9  2019 .profile
drwx------ 2 root root   37 May 21 04:57 .ssh
drwxr-x--- 5 root root  181 May 21 05:15 .vscode-server
-r

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
